# Module 092: Concurrency: Multiprocessing

Phase 10: Concurrency & Internals | Duration: 2.5 hours

## Process Basics

In [ ]:
import multiprocessing
import os
from typing import List

def worker(name: str) -> None:
    print(f"Process {name} (PID: {os.getpid()}) running")

processes: List[multiprocessing.Process] = []
for i in range(4):
    p = multiprocessing.Process(target=worker, args=(f"P-{i}",))
    p.start()
    processes.append(p)

for p in processes:
    p.join()

print(f"CPU count: {multiprocessing.cpu_count()}")

## Bypassing the GIL with Processes

Processes each have their own GIL, enabling true parallel execution for CPU-bound tasks.

In [ ]:
import multiprocessing
import time
from typing import List

def cpu_heavy(n: int) -> int:
    total: int = 0
    for i in range(n):
        total += i ** 2
    return total

N: int = 30000000

# Sequential
start: float = time.time()
cpu_heavy(N)
cpu_heavy(N)
sequential: float = time.time() - start
print(f"Sequential: {sequential:.2f}s")

# Parallel with processes
start = time.time()
processes = [multiprocessing.Process(target=cpu_heavy, args=(N,)) for _ in range(2)]
for p in processes:
    p.start()
for p in processes:
    p.join()
parallel: float = time.time() - start
print(f"Parallel (multiprocessing): {parallel:.2f}s")

## Process Pool

In [ ]:
from multiprocessing import Pool, cpu_count
from typing import List

def square(n: int) -> int:
    return n * n

numbers: List[int] = list(range(20))

with Pool(processes=cpu_count()) as pool:
    results: List[int] = pool.map(square, numbers)

print(f"Input: {numbers}")
print(f"Squared: {results}")
print(f"Using {cpu_count()} processes")

## Shared Memory: Value and Array

In [ ]:
from multiprocessing import Process, Value, Array
from typing import List

def increment_shared(val: Value, arr: Array) -> None:
    val.value += 1
    for i in range(len(arr)):
        arr[i] *= 2

shared_val = Value('i', 0)
shared_arr = Array('d', [1.0, 2.0, 3.0])

processes = [Process(target=increment_shared, args=(shared_val, shared_arr))
             for _ in range(5)]
for p in processes:
    p.start()
for p in processes:
    p.join()

print(f"Shared value: {shared_val.value}")
print(f"Shared array: {list(shared_arr)}")

## IPC with Queue and Pipe

In [ ]:
from multiprocessing import Process, Queue, Pipe
from typing import List

def sender(q: Queue) -> None:
    for i in range(5):
        q.put(f"Message-{i}")
    q.put(None)

def receiver(q: Queue) -> None:
    while True:
        msg = q.get()
        if msg is None:
            break
        print(f"Received: {msg}")

q: Queue = Queue()
p1 = Process(target=sender, args=(q,))
p2 = Process(target=receiver, args=(q,))
p1.start()
p2.start()
p1.join()
p2.join()
print("Queue IPC complete")

## Multiprocessing vs Threading Decision

| Feature | Threading | Multiprocessing |
|---------|-----------|-----------------|
| GIL limit | Yes | No (separate GIL per process) |
| Best for | I/O-bound tasks | CPU-bound tasks |
| Memory | Shared (risk of races) | Separate (higher overhead) |
| Start time | Fast | Slower |
| IPC | Queue/Lock (built-in) | Queue/Pipe/Shared memory |
| Crash isolation | Shared (one crash kills all) | Separate |